# 05 — Lane head & loss upgrade: point-based DETR lane (Stage 2)

**Innovation axis:** replace the YOLOPv2 **binary lane mask** with a
**set of lane polylines** predicted via a DETR-style query head and
trained with a Hungarian-matched geometric + raster loss.

This notebook **never touches the YOLOPv2 baseline** (`yolopv2_baseline.py`,
`yolopv2_vehicle_lane_baseline.yaml`). It trains a separate model,
`YOLOPv2-DETRLane`, with its own checkpoint directory. Switch between
the two by re-running notebook 02 or this one — they do not collide.

**Why this is compatible with the anchor detection head.** The two
heads share only the ELAN backbone + PAN neck. Detection is still the
dense anchor IDetect head (Y_baseline); lane is now a sparse-query
transformer. Parameter sets are fully disjoint except at the trunk, so
gradient conflict is limited to the shared encoder — which is the
point of multi-task learning. To reduce that residual conflict we use:
  * per-task residual adapters after each PAN scale (DETR_GeoLane idea)
  * a staged training gate (detection-only warmup, then joint)
  * a curriculum between geometric and soft-raster lane losses

All three are supported by the 2024 multi-task-learning literature
(PCGrad, CAGrad, expert-squads, task adapters). See AUDIT.md and
STAGE2_PLAN.md for references.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm opencv-python-headless tensorboard scipy

In [ ]:
import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
print('cwd:', os.getcwd())

In [ ]:
import logging, math
import numpy as np
import torch
from torch.cuda import amp
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as T

from lib.config import cfg
from lib.models import get_net
from lib.core import get_loss_detrlane
from lib.dataset import BddDatasetPoints
from lib.utils.drive_dataset import ensure_local_dataset_from_drive, find_raw_bdd_root, find_lane_polygon_jsons

In [ ]:
# ── Config ──
cfg.defrost()
cfg.merge_from_file(os.path.join(REPO_ROOT, 'configs', 'yolopv2_detrlane_vehicle_lane.yaml'))

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)
lane_jsons = find_lane_polygon_jsons(RAW_BDD_ROOT)

cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = os.path.join(RAW_BDD_ROOT, 'images', '100k')
cfg.DATASET.LABELROOT = os.path.join(RAW_BDD_ROOT, 'labels', '100k')
cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')  # unused by points dataset but kept for path resolver
cfg.DATASET.LANE_JSON_TRAIN = lane_jsons['train'] or ''
cfg.DATASET.LANE_JSON_VAL   = lane_jsons['val']   or ''

os.makedirs(cfg.DRIVE.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.DRIVE.METRICS_DIR, exist_ok=True)
tb_log_dir = os.path.join(cfg.DRIVE.ROOT, 'yolop_vehicle_lane', 'tb_logs', 'yolopv2_detrlane')
os.makedirs(tb_log_dir, exist_ok=True)
cfg.freeze()

print('MODEL.NAME         :', cfg.MODEL.NAME)
print('LANE.NUM_QUERIES   :', cfg.LANE.NUM_QUERIES, '| NUM_POINTS:', cfg.LANE.NUM_POINTS)
print('LANE_JSON_TRAIN    :', cfg.DATASET.LANE_JSON_TRAIN)
print('LANE_JSON_VAL      :', cfg.DATASET.LANE_JSON_VAL)
print('DET_ONLY_EPOCHS    :', cfg.LANE.DET_ONLY_EPOCHS)
print('checkpoints        :', cfg.DRIVE.CHECKPOINT_DIR)

In [ ]:
# ── Datasets ──
train_ds = BddDatasetPoints(cfg, is_train=True,  inputsize=640, transform=T.ToTensor())
val_ds   = BddDatasetPoints(cfg, is_train=False, inputsize=640, transform=T.ToTensor())

train_loader = DataLoader(train_ds, batch_size=cfg.TRAIN.BATCH_SIZE_PER_GPU,
                          shuffle=True, num_workers=cfg.WORKERS,
                          pin_memory=cfg.PIN_MEMORY, collate_fn=train_ds.collate_fn)
val_loader = DataLoader(val_ds, batch_size=cfg.TEST.BATCH_SIZE_PER_GPU,
                        shuffle=False, num_workers=cfg.WORKERS,
                        pin_memory=cfg.PIN_MEMORY, collate_fn=val_ds.collate_fn)
print(f'train: {len(train_ds)} | val: {len(val_ds)}')

In [ ]:
# ── Model + loss + optimizer ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_net(cfg).to(device)
model.gr = 1.0
model.names = cfg.MODEL.VEHICLE_CLASSES
model.set_det_only_epochs(int(cfg.LANE.DET_ONLY_EPOCHS))

criterion = get_loss_detrlane(cfg, device)

if cfg.TRAIN.OPTIMIZER == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.TRAIN.LR0, betas=(cfg.TRAIN.MOMENTUM, 0.999))
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.TRAIN.LR0, momentum=cfg.TRAIN.MOMENTUM,
                                weight_decay=cfg.TRAIN.WD, nesterov=cfg.TRAIN.NESTEROV)
for pg in optimizer.param_groups:
    pg['initial_lr'] = cfg.TRAIN.LR0
lf = lambda x: ((1 + math.cos(x * math.pi / cfg.TRAIN.END_EPOCH)) / 2) * (1 - cfg.TRAIN.LRF) + cfg.TRAIN.LRF
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)
scaler = amp.GradScaler(enabled=device.type != 'cpu')
print(f'Model params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M')

In [ ]:
# ── Resume + training loop ──
logger = logging.getLogger('detrlane')
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
logger.setLevel(logging.INFO)
writer = SummaryWriter(tb_log_dir)

start_epoch = 0
ckpt_path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'latest.pth')
if os.path.exists(ckpt_path) and cfg.AUTO_RESUME:
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state_dict'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    logger.info(f'Resumed from epoch {start_epoch}')

num_batch = len(train_loader)

for epoch in range(start_epoch, cfg.TRAIN.END_EPOCH):
    model.train()
    model.set_epoch(epoch)
    criterion.set_epoch(epoch)
    for i, (imgs, targets, paths, shapes) in enumerate(train_loader):
        imgs = imgs.to(device, non_blocking=True)
        targets = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in targets.items()}
        with amp.autocast(enabled=device.type != 'cpu'):
            out = model(imgs)
            loss, info = criterion(out, targets, model)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if i % cfg.PRINT_FREQ == 0:
            logger.info(f'ep{epoch} [{i}/{num_batch}] loss={loss.item():.4f} | det={info["det_total"]:.3f} lane={info["lane_total"]:.3f}')
            writer.add_scalar('train/loss', loss.item(), epoch * num_batch + i)
            writer.add_scalar('train/det',  info['det_total'],  epoch * num_batch + i)
            writer.add_scalar('train/lane', info['lane_total'], epoch * num_batch + i)
    scheduler.step()

    # save each epoch
    torch.save({
        'epoch': epoch,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
    }, ckpt_path)
    logger.info(f'checkpoint -> {ckpt_path}')

writer.close()
print('Training complete.')

In [ ]:
# ── Quick visual check: overlay predicted polylines ──
import matplotlib.pyplot as plt
model.eval()
with torch.no_grad():
    imgs, targets, paths, shapes = next(iter(val_loader))
    out = model(imgs.to(device))
pred_pts = out['lane']['pred_points'].cpu().numpy()          # [B, Q, P, 2]
pred_exist = out['lane']['exist_logits'].sigmoid().cpu().numpy()  # [B, Q, 1]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, ax in enumerate(axes.flat):
    if i >= imgs.shape[0]:
        break
    im = imgs[i].permute(1, 2, 0).numpy()
    ax.imshow(np.clip(im, 0, 1))
    H, W = im.shape[:2]
    for q in range(pred_pts.shape[1]):
        if pred_exist[i, q, 0] < 0.35:
            continue
        xy = pred_pts[i, q]
        ax.plot(xy[:, 0] * W, xy[:, 1] * H, '-', linewidth=2)
    # GT
    for q in range(targets['lane_points'].shape[1]):
        if targets['lane_existence'][i, q] < 0.5:
            continue
        xy = targets['lane_points'][i, q].cpu().numpy()
        ax.plot(xy[:, 0] * W, xy[:, 1] * H, '--', color='white', linewidth=1, alpha=0.7)
    ax.set_title(f'val[{i}]')
    ax.axis('off')
plt.tight_layout()
out_png = os.path.join(cfg.DRIVE.METRICS_DIR, 'lane_overlay_preview.png')
plt.savefig(out_png, dpi=120, bbox_inches='tight')
print('Saved overlay ->', out_png)